In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import torch

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "Enable a GPU runtime first."

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", gpu_name)
print(f"VRAM: {gpu_gib:.1f} GiB")
print("CUDA:", torch.version.cuda)

assert "A100" in gpu_name, f"Expected A100, received {gpu_name}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GiB
CUDA: 12.8


In [2]:
from pathlib import Path
import os
import subprocess

REPO = Path("/content/MailoHLS")
COMMIT = input("Paste the post-fix MailoHLS commit SHA: ").strip()

assert len(COMMIT) == 40

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ElenaVouvali/MailoHLS.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run(["git", "fetch", "--all"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)

actual = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert actual == COMMIT
os.chdir(REPO)
print("Checked out:", actual)

Paste the post-fix MailoHLS commit SHA: 1221303a3fc8b028a4bbad993ba203ac22b1b59d
Checked out: 1221303a3fc8b028a4bbad993ba203ac22b1b59d


In [3]:
import subprocess
import sys

packages = [
    "transformers==4.51.0",
    "peft==0.18.1",
    "accelerate==1.13.0",
    "bitsandbytes==0.49.2",
    "einops==0.8.2",
    "einops-exts==0.0.4",
    "safetensors==0.7.0",
    "sentencepiece",
    "pytest",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages],
    check=True,
)

import inspect
import transformers
import peft
import accelerate
import bitsandbytes
from peft import LoraConfig

assert "trainable_token_indices" in inspect.signature(
    LoraConfig
).parameters

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

torch: 2.11.0+cu128
transformers: 4.51.0
peft: 0.18.1
accelerate: 1.13.0
bitsandbytes: 0.49.2


In [5]:
from pathlib import Path
import hashlib
import json
import shutil
from collections import Counter

DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/MailoHLS/LLM_data"
)
LOCAL_DATASET_ROOT = REPO / "artifacts/llm"

required_files = [
    "mailohls_sft.jsonl",
    "mailohls_sft.sources.json",
    "mailohls_sft.manifest.json",
]

LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

for filename in required_files:
    source = DRIVE_DATASET_ROOT / filename
    destination = LOCAL_DATASET_ROOT / filename

    assert source.is_file(), f"Missing required dataset file: {source}"
    shutil.copy2(source, destination)
    print("Copied:", destination)

DATASET = LOCAL_DATASET_ROOT / "mailohls_sft.jsonl"

digest = hashlib.sha256(DATASET.read_bytes()).hexdigest()

rows = []
with DATASET.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        assert "kernel_name" in row, line_number
        rows.append(row)

print("dataset:", DATASET)
print("sha256:", digest)
print("rows:", len(rows))
print("devices:", Counter(str(r.get("device")) for r in rows))
print("kernels:", len({r["kernel_name"] for r in rows}))

assert rows



Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl
Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.sources.json
Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.manifest.json
dataset: /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl
sha256: 0a235f79abd4b065c0cff0e943ab91ca20408cfe69358b3b61051a98216d6a14
rows: 150026
devices: Counter({'xczu7ev-ffvc1156-2-e': 78571, 'xcu200-fsgd2104-2-e': 71455})
kernels: 55


In [6]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "LLM_branch/tests/test_structural_xattn.py",
        "LLM_branch/tests/test_stage1_adapter_roundtrip.py",
    ],
    cwd=REPO,
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "py_compile",
        "LLM_branch/train/train_SFT_xattn_new.py",
        "LLM_branch/inference/eval_stage1_stage2_stage3.py",
    ],
    cwd=REPO,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'py_compile', 'LLM_branch/train/train_SFT_xattn_new.py', 'LLM_branch/inference/eval_stage1_stage2_stage3.py'], returncode=0)

In [10]:
from pathlib import Path
import os
import sys

RUN_ROOT = Path("/content/mailohls_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

SPLIT_JSON = RUN_ROOT / "stage1_stage2_family_split_s123.json"
SMOKE_DIR = RUN_ROOT / "stage1_pareto_adp_smoke_s123"
FULL_DIR = RUN_ROOT / "stage1_pareto_adp_full_s123"

if gpu_gib >= 70:
    batch_size = 2
    grad_accum = 4
else:
    batch_size = 1
    grad_accum = 8

print("batch_size:", batch_size)
print("grad_accum:", grad_accum)
print("effective batch:", batch_size * grad_accum)

ENV = os.environ.copy()
ENV["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
ENV["PYTHONHASHSEED"] = "0"
ENV["TOKENIZERS_PARALLELISM"] = "false"

# Add REPO to PYTHONPATH for subprocess to find custom modules
if "PYTHONPATH" in ENV:
    ENV["PYTHONPATH"] = f"{REPO}:{ENV['PYTHONPATH']}"
else:
    ENV["PYTHONPATH"] = str(REPO)

from huggingface_hub import model_info
from transformers import AutoConfig, AutoTokenizer

MODEL_ID = "deepseek-ai/deepseek-coder-6.7b-base"
MODEL_REVISION = model_info(MODEL_ID).sha

config = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)
assert config.max_position_embeddings >= 7168

AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Context:", config.max_position_embeddings)

COMMON = [
    sys.executable, "-u",
    "LLM_branch/train/train_SFT_xattn_new.py",

    "--run_mode", "single",
    "--objective", "PARETO_ADP",
    "--dataset", str(DATASET),

    "--split_mode", "family",
    "--val_families",
    "rodinia_pathfinder;machsuite_sort_radix",
    "--test_families",
    "serrano-kalman-filter",

    "--disable_structural_memory",
    "--best_dir_name", "best_custom_stage1",

    "--device_mode", "known",
    "--device_token_dropout", "0",

    "--resource_budget_mode", "random",
    "--random_budgets_per_case", "16",
    "--random_budget_min_frac", "0.10",
    "--min_feasible_candidates_per_budget", "3",
    "--candidate_pool_per_objective", "24",
    "--auto_frequency_fraction", "0",

    "--top_k", "1",
    "--goal_domination_penalty", "0.25",
    "--goal_max_dominated_gap", "0.12",
    "--min_supervised_sites", "2",
    "--min_site_coverage", "0.85",
    "--score_weight_min", "0.6",
    "--score_weight_power", "1.0",

    "--candidate_loss_weight", "0",
    "--candidate_sites_per_sample", "2",
    "--candidate_negatives_per_site", "2",
    "--candidate_max_prefix_tokens", "1536",
    "--candidate_keep_head_tokens", "256",

    "--max_length", "7168",
    "--batch_size", str(batch_size),
    "--grad_accum", str(grad_accum),
    "--num_workers", "2",
    "--group_by_length",
    "--gradient_checkpointing",

    "--lr_lora", "5e-5",
    "--lr_embed", "5e-5",
    "--lora_r", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",

    "--selection_num_val_kernels", "4",
    "--loss_chunk_t", "256",
    "--seed", "123",
]

batch_size: 1
grad_accum: 8
effective batch: 8
Model: deepseek-ai/deepseek-coder-6.7b-base
Revision: ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912
Context: 16384


In [11]:
import subprocess

assert not SMOKE_DIR.exists(), (
    f"{SMOKE_DIR} already exists; use a fresh run directory."
)
assert not SPLIT_JSON.exists(), (
    f"{SPLIT_JSON} already exists; verify it or use a fresh path."
)

LOG_PATH = RUN_ROOT / "stage1_pareto_adp_smoke_s123.log"

smoke_command = COMMON + [
    "--save_split_json", str(SPLIT_JSON),
    "--output_dir", str(SMOKE_DIR),
    "--epochs", "1",
    "--max_steps", "2",
    "--eval_steps", "1",
    "--save_steps", "1",
]

with LOG_PATH.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        smoke_command,
        cwd=REPO,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

returncode = process.wait()
if returncode != 0:
    raise RuntimeError(
        f"Stage-1 smoke failed with exit code {returncode}. "
        f"Full log: {LOG_PATH}"
    )

assert SPLIT_JSON.is_file()
assert (SMOKE_DIR / "best_custom_stage1").is_dir()
assert (SMOKE_DIR / "best_custom_stage1/training_contract.json").is_file()

print("Stage-1 smoke test passed.")

[TARGET-CONDITIONING] TargetAwareConfig(device_mode='known', adapt_device='', budget_mode='random', random_budgets_per_case=16, min_budget_frac=0.1, min_feasible_candidates=3, candidate_pool_per_objective=24, auto_frequency_fraction=0.0, min_auto_clock_count=2, strict_source_markers=True, seed=123)
[CUDA] using: NVIDIA A100-SXM4-40GB
[INFO] Loaded 150026 raw rows from /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl
[INFO] Raw rows per family (top 15): [('rodinia_knn', 19755), ('rodinia_streamcluster', 19141), ('rodinia_cfd_step_factor', 18866), ('rodinia_kmeans', 14532), ('rodinia_hotspot', 12230), ('rodinia_lavamd', 10968), ('rodinia_dilate', 10471), ('rodinia_backprop', 7739), ('serrano_kalman_filter', 6705), ('rodinia_pathfinder', 5647), ('machsuite_sort_radix', 4035), ('rodinia_lc_gicov', 3815), ('spcl_example', 3683), ('machsuite_stencil3d', 3459), ('machsuite_viterbi', 3329)]
[INFO] val_families: ['machsuite_sort_radix', 'rodinia_pathfinder']
[INFO] test_families: ['serrano_ka

In [ ]:
import subprocess

assert not FULL_DIR.exists(), (
    f"{FULL_DIR} already exists; resume intentionally or use a fresh path."
)

full_command = COMMON + [
    "--split_json", str(SPLIT_JSON),
    "--output_dir", str(FULL_DIR),
    "--epochs", "3",
    "--max_steps", "-1",
    "--eval_steps", "50",
    "--save_steps", "50",
]

subprocess.run(
    full_command,
    cwd=REPO,
    env=ENV,
    check=True,
)

BEST_STAGE1 = FULL_DIR / "best_custom_stage1"

required = [
    BEST_STAGE1 / "adapter_config.json",
    BEST_STAGE1 / "training_contract.json",
    BEST_STAGE1 / "tokenizer_config.json",
    BEST_STAGE1 / "best_selection_metrics.json",
]

for path in required:
    assert path.is_file(), path

assert (
    (BEST_STAGE1 / "adapter_model.safetensors").is_file()
    or (BEST_STAGE1 / "adapter_model.bin").is_file()
)

print("Best Stage 1 adapter:", BEST_STAGE1)

In [ ]:
import subprocess
import json
import statistics

VAL_CASES = (
    FULL_DIR
    / "selected_debug"
    / "val_selected_pareto_adp.jsonl"
)
EVAL_OUTPUT = FULL_DIR / "stage1_val_predictions.jsonl"

assert VAL_CASES.is_file(), VAL_CASES

eval_command = [
    sys.executable, "-u",
    "LLM_branch/inference/eval_stage1_stage2_stage3.py",

    "--stage", "stage1",
    "--adapter_dir", str(BEST_STAGE1),
    "--candidate_bank_dataset", str(DATASET),
    "--candidate_bank_exclude_families",
    "rodinia_pathfinder;machsuite_sort_radix;serrano-kalman-filter",
    "--input_jsonl", str(VAL_CASES),
    "--output_jsonl", str(EVAL_OUTPUT),
    "--max_prompt_tokens", "7168",
    "--score_reduction", "mean",
]

subprocess.run(
    eval_command,
    cwd=REPO,
    env=ENV,
    check=True,
)

predictions = [
    json.loads(line)
    for line in EVAL_OUTPUT.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

value_accuracy = statistics.mean(
    row["value_accuracy_over_expected"]
    for row in predictions
)

schema_rate = statistics.mean(
    float(row["schema_compliant"])
    for row in predictions
)

print("cases:", len(predictions))
print("mean per-slot value accuracy:", value_accuracy)
print("schema compliance:", schema_rate)

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import tarfile

EXPORT_ROOT = Path("/content/mailohls_stage1_export")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "freeze",
    ],
    stdout=(EXPORT_ROOT / "pip_freeze.txt").open("w"),
    check=True,
)

manifest = {
    "git_commit": COMMIT,
    "dataset_sha256": hashlib.sha256(
        DATASET.read_bytes()
    ).hexdigest(),
    "split_sha256": hashlib.sha256(
        SPLIT_JSON.read_bytes()
    ).hexdigest(),
    "gpu": gpu_name,
    "gpu_memory_gib": gpu_gib,
    "stage1_adapter": "best_custom_stage1",
}

(EXPORT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

LOCAL_ARCHIVE = Path(
    "/content/mailohls_stage1_pareto_adp_s123.tar.gz"
)

with tarfile.open(LOCAL_ARCHIVE, "w:gz") as archive:
    archive.add(
        BEST_STAGE1,
        arcname="best_custom_stage1",
    )
    archive.add(
        SPLIT_JSON,
        arcname="stage1_stage2_family_split_s123.json",
    )
    archive.add(
        EVAL_OUTPUT,
        arcname="stage1_val_predictions.jsonl",
    )
    archive.add(
        FULL_DIR / "selected_debug",
        arcname="selected_debug",
    )
    archive.add(
        EXPORT_ROOT,
        arcname="reproducibility",
    )

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/MailoHLS/Outputs"
)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

DRIVE_ARCHIVE = DRIVE_OUTPUT / LOCAL_ARCHIVE.name
shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

archive_sha = hashlib.sha256(
    DRIVE_ARCHIVE.read_bytes()
).hexdigest()

print("Saved:", DRIVE_ARCHIVE)
print("sha256:", archive_sha)